**Which vectorization**

In [ ]:
!pip install mlflow boto3 awscli

In [ ]:
# AWS Access Key ID
!aws configure

In [ ]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")

In [ ]:
# create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os


In [ ]:
# Removes all rows where the column clean_comment is empty
df=pd.read_csv("reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.shape

In [ ]:
from numpy import vectorize
# Setp 1: Function to run the experiment
def run_experiment(vectorizer_type,ngram_range,vectorizer_max_features,vectorizer_name):
  # Step 2: Vectorization
  if vectorizer_type== "BoW":
    vectorizer=CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
  else:
    vectorizer=TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

  X_train, X_test, y_train,y_test=train_test_split(df["clean_comment"],df["category"],test_size=0.2,random_state=42)

  X_train=vectorizer.fit_transform(X_train)
  X_test=vectorizer.transform(X_test)

  # Step 4: Define and train a Random Forest model
  with mlflow.start_run() as run:
    # Set tags for the experiment and run and add desciption
    mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
    mlflow.set_tag("experiment_type", "feature_engineering")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag("desciption", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

    # Log vectorizer parameters
    mlflow.log_params("vectorizer_type", vectorizer_type)
    mlflow.log_param("ngram_range", ngram_range)
    mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

    # Log Random Forest parameters
    n_estimators=200
    max_depth=15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # Initialize and train the model
    model= RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train,y_train)

    # Step 5: Make predictions and log metrics
    y_pred=model.predict(X_test)

    # Log accuracy
    accuracy=accuracy_score(y_test,y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Log classification report
    classification_rep=classification_report(y_test,y_pred,out_dict=True)
    for label, metrics in classification_rep.items():
       if isinstance(metrics,dict):
         for metric, value in metrics.items():
            mlflow.log_metric(f"{label}_{metric}",value)

    # Log confusion matrix
    conf_matrix=confusion_matrix(y_test,y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(conf_matrix,annot=True,fmt="d",cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifcat("confusion_matrix.png")
    plt.close()

    # Log the model
    mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6:Use different n-grams
ngram_ranges=[(1,1),(1,2),(1,3)]
max_features=5000

for ngram_range in ngram_ranges:
  run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

  run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


